# 🧠 Mood-Adaptive Daily Planner & Motivation Agent


>  **📌 Problem / UseCase : People frequently feel overwhelmed, low-energy, or unmotivated and waste time deciding what to do; they need a quick, personalized plan and an encouraging message to start their day.**



 
> **💡Solution: An agent that asks a simple set of user inputs (mood, energy level, time available, high-level goals) and returns:**
> *  **Mood-appropriate motivational message**
> *  **Prioritized, time-aware daily plan**
> *  **Short well-being micro-activity (1–5 mins)**




> **⚠️Risks : The assistant avoids clinical diagnosis or therapy and provides only general motivational content and safe micro-activities. Include disclaimers in the UI: not a replacement for professional help.**


**1. Configure your Gemini API Key**

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}")

**2. Import ADK components**

-- Use In-Memory

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types
import json

print("✅ ADK components imported successfully.")

# Define the Agent with ADK

 **3. Define the Agent**



In [ ]:
# Agent to generate mood plan
mood_planner_agent = Agent(
    name="mood_planner_agent",
    model="gemini-2.5-flash-lite",
    description="Mood-based planner agent.",
    instruction=(
        "You are a mood-adaptive daily planner. "
        "Always use the user's inputs: mood, energy level, time available, and goals. "
        "Return ONLY valid JSON with keys: motivation_message, daily_plan, micro_activity. "
        "Daily plan tasks must fit within the available time."
    ),
    tools=[]
)
print("✅ Mood Planner Agent defined.")

**4. Create Runner**
   Run the agent in memory

In [ ]:
runner = InMemoryRunner(agent=mood_planner_agent)

print("✅ Runner created.")

**5. Generate a plan**

> Inputs: mood, energy_level, time_available, high-level goals

> Output: JSON with keys: motivation_message, daily_plan, micro_activity


In [ ]:
import json
import re
async def generate_daily_plan(mood, energy_level, time_available, goals=None):
    goals_text = ", ".join(goals) if goals else "general productivity"

    prompt_text = f"""
You are a friendly productivity assistant.

User info:
Mood: {mood}
Energy level (1-5): {energy_level}
Time available: {time_available} minutes
Goals: {goals_text}

Task requirements:
1. Short mood-appropriate motivational message.
2. Prioritized, time-aware daily plan (tasks + durations) that TOTAL ≤ {time_available} minutes.
3. One micro-activity (1–5 min): breathing, stretching, or journaling.

Return ONLY valid JSON with keys:
- motivation_message
- daily_plan (list of tasks with 'task', 'duration_min', 'priority')
- micro_activity
"""

    # Run the agent asynchronously
    events = await runner.run_debug(prompt_text)
    last_event = list(events)[-1]  # convert generator to list
    raw_text = last_event.content.parts[0].text

    # Remove code fences if any
    cleaned_text = re.sub(r"```json|```", "", raw_text, flags=re.IGNORECASE).strip()

    # Parse JSON
    try:
        return json.loads(cleaned_text)
    except Exception as e:
        print("⚠️ Failed to parse JSON:", e)
        print("Raw output:", cleaned_text)
        return None


print("✅ Plan created.")

**6. Send a prompt and get an answer.**

In [ ]:
plan = await generate_daily_plan(
    mood="lazy",
    energy_level=2,
    time_available=60,
    goals=["cook", "car wash"]
)



# User Interface

**7. Add a UI for Demo**

**7.1 Install and Import dependencies for UI display**

In [ ]:
!pip install ipywidgets nest_asyncio
import nest_asyncio, asyncio
from IPython.display import display, Markdown
import ipywidgets as widgets

nest_asyncio.apply()
print("✅ UI Dependencies installed.")

**7.2 Synchronous Wrapper**

In [ ]:
# Synchronous wrapper
def get_daily_plan(mood, energy, time, goals_text):
    goals = [g.strip() for g in goals_text.split(',')] if goals_text else [] 
    return asyncio.get_event_loop().run_until_complete( generate_daily_plan(mood, int(energy), int(time), goals) )
    
print("✅ Synchronous wrapper defined.")

**7.3 Interactive UI**

In [ ]:

# Input widgets
mood_widget = widgets.Text(value='lazy', description='Mood:')
energy_widget = widgets.IntSlider(value=2, min=1, max=5, description='Energy:')
time_widget = widgets.IntText(value=60, description='Time (min):')
goals_widget = widgets.Text(value='cook, car wash', description='Goals:')
output_widget = widgets.Output()

# Priority color helper
def priority_color(priority):
    colors = {1: '#ff9999', 2: '#ffcc99', 3: '#ffff99'}
    return colors.get(priority, '#dddddd')

# Button callback
def on_button_click(b):
    output_widget.clear_output()
    plan = get_daily_plan(mood_widget.value, energy_widget.value, time_widget.value, goals_widget.value)
    if plan is None:
        with output_widget:
            display(widgets.HTML("<p style='color:red;'>Failed to generate plan. Check agent output.</p>"))
        return
    
    with output_widget:
        # Motivation Section
        display(widgets.HTML(f"""
        <div style='background:#d0f0c0;padding:10px;border-radius:10px;margin-bottom:10px;'>
            <b style='font-size:16px;'>Motivation</b>
            <p>{plan['motivation_message']}</p>
        </div>
        """))


        # Tasks Section with checkboxes
        task_rows = []
        for t in plan['daily_plan']:
            
            cb = widgets.Checkbox(value=False, indent=False)
            
            label = widgets.HTML(
                f"<div style='background:{priority_color(t['priority'])};"
                f"padding:10px;margin:2px;border-radius:8px;'>"
                f"<b>{t['task']}</b> - {t['duration_min']} min (Priority {t['priority']})</div>"
                )
            
            row = widgets.HBox([cb, label])
            task_rows.append(row)
        
            tasks_box = widgets.VBox(task_rows)
        
        tasks_card = widgets.HTML(
            f"<div style='background:#f0f0f0;padding:10px;border-radius:10px;margin-bottom:10px;'>"
            f"<b style='font-size:16px;'>Your Tasks</b></div>"
        )
        display(tasks_card, tasks_box)
    
        
        # Micro Activity Section
        display(widgets.HTML(f"""
        <div style='background:#c0d0f0;padding:10px;border-radius:10px;'>
            <b style='font-size:16px;'>Micro Activity</b>
            <p>{plan['micro_activity']}</p>
        </div>
        """))

# Button
button = widgets.Button(description="Generate Plan", button_style='success')
button.on_click(on_button_click)

# Layout
ui = widgets.VBox([mood_widget, energy_widget, time_widget, goals_widget, button, output_widget])
display(ui)


# Agent Deployment

**8. Deployment Setup**

**8.1 Import Components**

In [ ]:
import os
import random
import time
import vertexai
from kaggle_secrets import UserSecretsClient
from vertexai import agent_engines

print("✅ Imports completed successfully")

**8.2  Add Cloud Credentials**

In [ ]:
# Set up Cloud Credentials in Kaggle
user_secrets = UserSecretsClient()
user_credential = user_secrets.get_gcloud_credential()
user_secrets.set_tensorflow_credential(user_credential)

print("✅ Cloud credentials configured")

**8.3 Set project ID**

In [ ]:
## Set your PROJECT_ID
PROJECT_ID = "animated-factor-478816-t2"  # TODO: Replace with your project ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

if PROJECT_ID == "your-project-id" or not PROJECT_ID:
    raise ValueError("⚠️ Please replace 'your-project-id' with your actual Google Cloud Project ID.")

print(f"✅ Project ID set to: {PROJECT_ID}")

# Creating Agent with ADK

**8.4 Create agent directory**

In [ ]:
## Create simple agent - all code for the agent will live in this directory
!mkdir -p sample_agent

print(f"✅ Sample Agent directory created")

**8.5 Create requirements file**

In [ ]:
%%writefile sample_agent/requirements.txt

google-adk
opentelemetry-instrumentation-google-genai

**8.6 Create environment configuration**

In [ ]:
%%writefile sample_agent/.env

# https://cloud.google.com/vertex-ai/generative-ai/docs/learn/locations#global-endpoint
GOOGLE_CLOUD_LOCATION="global"

# Set to 1 to use Vertex AI, or 0 to use Google AI Studio
GOOGLE_GENAI_USE_VERTEXAI=1

**8.7 Create agent code**

In [ ]:
%%writefile sample_agent/agent.py
from google.adk.agents import Agent
import vertexai
import os

vertexai.init(
    project=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ["GOOGLE_CLOUD_LOCATION"],
)



import json

def generate_daily_plan(mood: str, energy_level: int, time_available: int, goals=None) -> dict:
    """
    Prepares structured user context for generating a daily plan.

    This is a TOOL the agent can call. Unlike the previous version,
    it does NOT run the LLM. It simply returns structured input data
    that the agent’s model will use to create the plan.

    Args:
        mood (str): User's current mood (e.g., "lazy", "focused").
        energy_level (int): User-reported energy (1–5 scale).
        time_available (int): Total minutes available.
        goals (list, optional): Optional high-level goals.

    Returns:
        dict: Structured context for the agent, containing:
            - mood
            - energy_level
            - time_available
            - goals (list)
            - prompt_instructions (what the model should do)
    """

    return {
        "status": "success",
        "mood": mood,
        "energy_level": energy_level,
        "time_available": time_available,
        "goals": goals or [],
        "prompt_instructions": {
            "task": "Create a motivational message, daily plan, and micro-activity.",
            "rules": [
                f"Total time must NOT exceed {time_available} minutes.",
                "Plan should include priorities and duration for each task.",
                "Provide one micro-activity (1–5 minutes).",
                "Return ONLY valid JSON."
            ]
        }
    }

print("✅ Daily plan tool loaded.")
root_agent = Agent(
    name="daily_planner",
    model="gemini-2.5-flash-lite",  # Fast, cost-effective model
    description="A friendly productivity assistant that generates daily plans based on user mood, energy, time, and goals.",
    instruction="""
    You are a helpful productivity assistant. When users ask for a daily plan:
      1. Collect mood, energy level, available time, and optional goals.
      2. Use the generate_daily_plan_tool to create a structured daily plan.
      3. Respond in a friendly, motivating tone.
      4. Always return valid JSON with keys 'motivation_message', 'daily_plan', and 'micro_activity'.
    """,
    tools=[generate_daily_plan]
)

# Deploy to Agent Engine

**8.8 Create deployment configuration**

In [ ]:
%%writefile sample_agent/.agent_engine_config.json
{
    "min_instances": 0,
    "max_instances": 1,
    "resource_limits": {"cpu": "1", "memory": "1Gi"}
}

**8.9 Select deployment region**

In [ ]:
regions_list = ["europe-west1", "europe-west4", "us-east4", "us-west1"]
deployed_region = random.choice(regions_list)

print(f"✅ Selected deployment region: {deployed_region}")

**8.10 Deploy the agent**

In [ ]:
!adk deploy agent_engine --project=$PROJECT_ID --region=$deployed_region sample_agent --agent_engine_config_file=sample_agent/.agent_engine_config.json

# Retrieve and Test Your Deployed Agent

**8.11 Retrieve the deployed agent**

In [ ]:
# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=deployed_region)

# Get the most recently deployed agent
agents_list = list(agent_engines.list())
if agents_list:
    remote_agent = agents_list[0]  # Get the first (most recent) agent
    client = agent_engines
    print(f"✅ Connected to deployed agent: {remote_agent.resource_name}")
else:
    print("❌ No agents found. Please deploy first.")

**8.12 Test the deployed agent**

In [ ]:
async for event in remote_agent.async_stream_query(
    message="""
        Create a daily plan.
        Mood: lazy
        Energy: 2
        Time available: 60 minutes
        Goals: cook, car wash
    """,
    user_id="user_42",
):
    print(event)


# Clean Up

In [ ]:
agent_engines.delete(resource_name=remote_agent.resource_name, force=True)

print("✅ Agent successfully deleted")